# NB07 — Correctness Against the C++ Oracle
「教學版同生產版一唔一致？」呢課答呢個問題，順便講吓兩個實作嘅差異哲學。

In [1]:
import sys, pathlib
# repo-root relative imports so the notebook runs from anywhere
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / 'edu' / 'sl_edu').exists()) \
       if not (pathlib.Path.cwd() / 'edu' / 'sl_edu').exists() else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'edu'))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 4)
DATA = ROOT / 'data'


## 1. Golden values（最強嘅對照）
C++ gtest 對同一數據集斷言咗具體 pixel 值。Python 版行同一 decode 鏈，
必須命中相同值——呢個係 pixel-exact 嘅跨語言對照。

In [2]:
from sl_edu import decode, oracle, patterns

imgs = oracle.load_shift_graycode(ROOT)
wrapped = decode.wrapped_phase(imgs[:4], 4)
conf = decode.confidence_map(imgs[:4])
floor70 = decode.floor_map(imgs[4:], conf, wrapped, 32, 70.0)
abs70 = decode.unwrap(wrapped, floor70, conf, 70.0)

anchors = [
    ('floor[453][700] == 17', floor70[453][700] == 17),
    ('|unwrap[460][653] - 103.75| <= 0.1', abs(abs70[460][653] - 103.75) <= 0.1),
]
gen = patterns.generate(1920, 1080, 4, 32)
anchors.append(('generated imgs[6][400][215] == 255', gen[6][400][215] == 255))

for name, ok in anchors:
    print(('PASS' if ok else 'FAIL'), '-', name)
assert all(ok for _, ok in anchors)

PASS - floor[453][700] == 17
PASS - |unwrap[460][653] - 103.75| <= 0.1
PASS - generated imgs[6][400][215] == 255


## 2. 已知差異清單（全部記錄喺 tests 入面）
| 差異 | 原因 | 處理 |
|---|---|---|
| wrap 邊界 ±1px | atan2 分子數值為零，uint8 量化擲毫 | test 放寬 + 註明 |
| 右邊緣 30 列 | 平移格雷碼 wrap-around（C++ 一樣有） | 文件化，實戰 crop |
| cv2.structured_light 比對唔上 | OpenCV 5 生成慣例同 SLMaster 唔同 | nb 討論，唔做 anchor |
| 性能 | numpy loop vs SIMD/parallel_for/CUDA | 教學冇所謂，下面量度 |

## 3. 性能對照（教學版嘅代價）

In [3]:
import time

t0 = time.perf_counter()
_ = decode.wrapped_phase(imgs[:4], 4)
t1 = time.perf_counter()
print(f'numpy wrapped_phase (1280x1024): {(t1-t0)*1000:.0f} ms')
print('C++ equivalent: ~1 ms (SIMD) — ~100x slower here, and that is fine.')
print()
print('Takeaway: 概念正確先行，性能係工程問題。')
print('C++ 嘅存在理由 = 實時 + 硬件控制；Python 嘅存在理由 = 可讀 + 可改。')

numpy wrapped_phase (1280x1024): 19 ms
C++ equivalent: ~1 ms (SIMD) — ~100x slower here, and that is fine.

Takeaway: 概念正確先行，性能係工程問題。
C++ 嘅存在理由 = 實時 + 硬件控制；Python 嘅存在理由 = 可讀 + 可改。


## 課程總結
1. **nb01** 編碼：正弦相位 + 格雷碼，邊界錯開半週期
2. **nb02** 捕捉：時間協議同步，virtual 模式冇硬件都跑到
3. **nb03** 解碼：atan2 → XOR gray → 邊界修正 → unwrap
4. **nb04** 標定：投影儀當相機教，cv2 解內外參
5. **nb05** 三角化：射線 ∩ 平面 = 3D 點 → 點雲
6. **nb06** live：webcam + 芒就係一台 3D 掃描儀
7. **nb07** 對照：pixel-exact vs C++ golden values

**白盒嘅價值**：每一環你都可以停低、plot、改參數、再嚟過。
RealSense 將以上全部收埋喺一粒 ASIC——快，但你學唔到嘢。